# BPI Challenge 2020: travel permission, expenses and payment

The improvement goal is to make expense decisions and payments easier to complete while preserving appropriate authorisation. Five linked extracts answer different questions: domestic declarations, international declarations, prepaid costs, payment requests and permits. Their records must not be summed as if they were five independent populations.

The analysis follows [PM²](https://vdaalst.com/publications/p824.pdf): Planning, Extraction, Data Processing, Mining and Analysis, Evaluation, and Process Improvement and Support. Each stage leaves a journal entry and an evidence question. Scores, priorities and sensitivity calculations come from WISE; this notebook selects questions and presents their results. No numerical finding is assumed before the source log has been analysed.

The [owner’s source description](https://icpmconference.org/2020/bpi-challenge/) distinguishes domestic trips without prior permission from international trips requiring an approved permit before arrangements. A permit extract includes related declaration and prepaid events; payment requests have a separate non-travel purpose. The source also distinguishes the limited 2017 pilot from the broader 2018 process. Roles replace personal identities, amounts are anonymised, and trip dates are estimates.

**Planning question.** Where are avoidable submission corrections, reviewer delays and payment-handling waits? Which patterns deserve a different response because they concern permits rather than payable declarations? Compare related process types separately, then use complete case evidence to decide whether a delay reflects an incomplete claim, necessary approval or avoidable queueing.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import wise
from bpic_analysis import (
    challenge, summary, norm_table, criterion_summary, activity_summary,
    context_profiles, priorities, plot_overview, case_evidence,
    threshold_sensitivity, view_agreement, integrity, journal, save_outputs,
)

DATA_ROOT = os.environ.get("WISE_PAPER_DATA")
EXPORT_RESULTS = False

## 1. Extraction: identify the population and observation boundary

Read each source from the supplied data folder. The summary records actual source populations and assessment status. Retain every observation of selected cases, including records outside a selected start-month cohort. A context extract without an event sequence is identified separately; it is not assigned a process score.

In [ ]:
runs = challenge(2020, data_folder=DATA_ROOT)
display(summary(runs))
scored_runs = {name: run for name, run in runs.items() if run.get("result") is not None}
display(journal(runs))

## 2. Data processing: check the meaning of observed activities

One permit can include several related documents. A repeated submission can be a correction or a different valid document; a rejection may perform a necessary control. A first submission-to-first positive approval at or after that submission interval is an initial-response diagnostic, not approval conformance for every document cycle. `Payment Handled` is an observed handling milestone, not independent proof of bank settlement. Permits do not themselves have a payment obligation.

The tables below distinguish event occurrences from cases with an observation. Counts are descriptive evidence; they are not effort, root causes or success rates.

In [ ]:
ACTIVITY_ANCHORS = ['Declaration SUBMITTED by EMPLOYEE', 'Declaration REJECTED by ADMINISTRATION', 'Declaration FINAL_APPROVED by SUPERVISOR', 'Request For Payment SUBMITTED by EMPLOYEE', 'Request For Payment FINAL_APPROVED by SUPERVISOR', 'Permit SUBMITTED by EMPLOYEE', 'Permit FINAL_APPROVED by SUPERVISOR', 'Request Payment', 'Payment Handled', 'Send Reminder']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(activity_summary(run, labels=ACTIVITY_ANCHORS))
    display(activity_summary(run).head(25))

## 3. Translate the business goal into a declared norm

The native diagnostic norms examine observed resubmissions, reviewer rejection signals, first positive approval timing and payment handling; the permit norm uses reminders instead of a permit payment requirement. International permission before arrangements is a published process requirement, but its implementation needs valid document links, scope and actual arrangement dates. A first-response lag does not enforce that requirement. The source’s two-calendar-month declaration period also cannot be replaced with an arbitrary 60-day lag against estimated trip dates.

Read the criterion, scope, thresholds and view weights before interpreting a score. For `Singularity`, `k` is the allowed count and `K` is the additional count range over which severity increases; `K` is not an upper allowed count. Native lag matches the first response at or after the first activation, allowing equal timestamps. Pending questions are shown so that an unsupported business requirement cannot disappear behind a convincing chart.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(norm_table(run))
    display(run["specification"].get("pending_checks", []))

## 4. Establish the denominator and evidence coverage

Check evidence availability separately for approval and payment criteria. Never treat a permit without payment as an unpaid declaration. An employee rejection can legitimately terminate a request; missing later payment evidence must be interpreted within the correct document route. Missing roles or undefined resources limit attribution. Evidence gaps should be a separate investigation list rather than silent low severity.

`evaluated_cases` and `in_scope_cases` show different denominators. Mean severity refers to usable evidence; an unavailable observation is not a zero violation. A high mean severity with narrow coverage deserves a different response from a widespread, well-observed problem.

In [ ]:
criterion_tables = {}
for name, run in scored_runs.items():
    criterion_tables[name] = criterion_summary(run)
    display(Markdown(f"### {name}"))
    display(criterion_tables[name])

## 5. Mining and analysis: compare stakeholder priorities

A timeliness lead may point to approval or payment handling; a rework lead may point to resubmission or reviewer rejection. The same score does not have the same business meaning in a permit and a declaration. Compare the view change within each extraction, then inspect the source population and criteria before making any cross-extract statement.

Priority combines scored case volume with a deficit in conformance score relative to the scored reference population. A higher conformance score means closer adherence to the declared norm; higher criterion severity means more deviation. The first rows show where to investigate, not which employee or process is at fault. An unranked or below-reference slice can still contain important individual cases.

In [ ]:
view_backlogs = {}
for name, run in scored_runs.items():
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        view_backlogs[(name, view)] = backlog
        display(Markdown(f"### {name} — {view}"))
        display(backlog.table.head(10))

## 6. Visualise severity, evidence and investigation order

The overview uses computed library results. Read the severity and coverage panels together, then compare their message with the priority list. A frequent activity is not automatically an important failure; a rare signal should retain access to its supporting case.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)
    table = criterion_tables[name]
    if not table.empty:
        figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        plotted = table.dropna(subset=["evidence_coverage", "mean_severity"])
        for row in plotted.itertuples(index=False):
            axis.scatter(row.evidence_coverage, row.mean_severity, s=85,
                         label=row.criterion_id)
        axis.legend(title="Criterion", loc="center left", bbox_to_anchor=(1, 0.5))
        axis.set(xlim=(-0.03, 1.08), ylim=(-0.03, 1.08),
                 xlabel="Evidence coverage within business scope",
                 ylabel="Mean severity within usable evidence",
                 title="Does a severe signal also have broad usable evidence?")
        display(figure)
        plt.close(figure)

## 7. Explain the signed layer drivers

Positive rework and negative timing contributions can coexist: a corrected declaration may still be approved quickly. A timing-heavy slice may have clean submissions but wait for an authorised reviewer. Keep those explanations separate so that an improvement does not remove a useful control or send more reminders to the wrong person.

Positive contributions increase the aggregate gap; negative contributions offset it. WISE keeps the signs and clips only the final sum. Do not remove the offsets to make a problem look larger.

In [ ]:
for (name, view), backlog in view_backlogs.items():
    result = scored_runs[name]["result"]
    display(Markdown(f"### {name} — {view}"))
    display(wise.layer_contributions(result, backlog).reindex(backlog.table.head(10).index))
    display(wise.layer_drivers(result, backlog).reindex(backlog.table.head(10).index))

## 8. Test thresholds, preferences and small-slice sensitivity

Compare cut-point scenarios within each process type. An approval lead that persists across scenarios is worth a case review, but threshold stability is not causal evidence. Compare shrinkage before prioritising a small cost centre or project. Preference sensitivity can reveal that financial administration and travellers need different follow-up actions from the same evidence.

These are declared scenarios, not estimates of an optimal policy. The timing scenario scales both the allowed lag and its severity width. The occurrence scenario keeps the allowed count `k` fixed and scales the additional severity width `K`; it does not halve or double the allowed count. The overlap measure concerns positive-priority shortlist membership and is undefined when both lists are empty. Low agreement is a prompt to explain the business trade-off.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(threshold_sensitivity(run, factors=(0.5, 1, 2)))
    display(view_agreement(run, k=10))
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    for gamma in (5, 20, 100):
        adjusted = priorities(run, view=view, gamma=gamma)
        display({"dataset": name, "view": view, "gamma": gamma,
                 "positive_top10_overlap": wise.top_k_overlap(reference, adjusted, k=10)})
        display(adjusted.table.head(10))

## 9. Compare complete cases in meaningful context

Use the whole declaration or permit as the case unit defined by its extraction. Organisational entity and project are case context only when stable or governed by an explicit change rule. Keep the 2017 pilot distinct from 2018 and retain later events when selecting a case-start cohort. Do not mistake the last recorded month for payment completion. Identifiers link records; they do not make overlapping extracts independent or identify individual employees.

A start month selects cases rather than splitting their events. Derived first/last observed months use UTC after timestamp conversion; they are not verified business-local calendar months. A business-calendar analysis needs the source timezone and its calendar rules. `last_observed_month` and `observed_span_band` describe observation boundaries. They are not verified finish-month or completion measures. A context field that changes within a case is not silently treated as a stable business attribute in the comparisons below.

The comparison below also gives the 2017 and 2018 **first-submission cohorts separate reference populations**. It uses the observed first document-submission timestamp in UTC, rather than an estimated trip marker. Check the source timezone before interpreting a boundary as a business-local year change. A selected case retains every later observation, including events in another year. This is an origin-cohort comparison under the same diagnostic norm, not proof that every event followed that year’s policy. Cases with unavailable submission dates remain in the complete-log analysis above.

In [ ]:
CONTEXT_KEYS = ['OrganizationalEntity', 'Permit OrganizationalEntity', 'Project', 'ProjectNumber', 'Permit ProjectNumber', 'Cost Type', 'BudgetNumber', 'case_start_month', 'observed_span_band']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(context_profiles(run, attributes=CONTEXT_KEYS))
    varying = run["log"].varying_case_attributes
    for attribute in CONTEXT_KEYS:
        if attribute not in run["result"].cases.columns:
            continue
        if varying.get(attribute, 0):
            display({"attribute": attribute, "comparison": "Context changes within cases; review the classification rule first"})
            continue
        display(Markdown(f"#### Whole-case grouping: {attribute}"))
        display(priorities(run, by=attribute, gamma=20).table.head(10))

SUBMISSION_LABELS = {
    "bpic2020_domestic": "Declaration SUBMITTED by EMPLOYEE",
    "bpic2020_international": "Declaration SUBMITTED by EMPLOYEE",
    "bpic2020_permit": "Permit SUBMITTED by EMPLOYEE",
    "bpic2020_prepaid": "Request For Payment SUBMITTED by EMPLOYEE",
    "bpic2020_payment": "Request For Payment SUBMITTED by EMPLOYEE",
}
period_backlogs = {}
for name, run in scored_runs.items():
    log = run["log"]
    submitted = log.first_ts(SUBMISSION_LABELS[name])
    for origin_year in (2017, 2018):
        selected = submitted.index[submitted.dt.year.eq(origin_year)]
        if len(selected) == 0:
            continue
        source_events = log.events
        selected_events = source_events.loc[source_events[log.case_col].isin(selected)]
        cohort_log = wise.EventLog(
            selected_events, case_col=log.case_col, activity_col=log.activity_col,
            timestamp_col=log.timestamp_col, case_attributes=log.case_attributes,
            utc=True, missing_timestamps="keep",
        )
        cohort_result = wise.score(cohort_log, run["norm"])
        cohort_context = run["result"].cases.reindex(cohort_result.scores.index)
        cohort_result = replace(cohort_result, cases=cohort_context)
        display(Markdown(f"#### {name}: first submitted in {origin_year}, separate reference"))
        for view in cohort_result.views:
            backlog = wise.prioritize(cohort_result, by="case_start_month", view=view, gamma=20)
            period_backlogs[(name, origin_year, view)] = backlog
            display(Markdown(f"**{view}**"))
            display(backlog.table.head(10))


## 10. Test how missing evidence changes the aggregation

Flat and layer-balanced scoring answer slightly different aggregation questions when evidence availability varies. Rescore the same complete cases through the library. Compare the resulting investigation order using identical slice keys and view; do not interpret a change in order as a newly observed process event.

In [ ]:
for name, run in scored_runs.items():
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    by = list(reference.spec.by)
    for mode in ("flat", "layer_balanced"):
        rescored = replace(wise.score(run["log"], run["norm"], mode=mode), cases=run["result"].cases)
        backlog = wise.prioritize(rescored, by=by, view=view, gamma=0)
        display({"dataset": name, "view": view, "mode": mode,
                 "positive_top10_overlap": wise.top_k_overlap(reference, backlog, k=10)})
        display(backlog.table.head(10))

## 11. Evaluation: inspect the full case behind a lead

For each process type, inspect a full case with a resubmission, rejection or timing signal. Trace who was expected to act, whether the rejection led to correction or a legitimate stop and whether several documents belong to one permit. A true policy review also needs the relevant document-cycle and linkage evidence; the native case score alone cannot establish a duplicate payment or unauthorised travel.

The witness retains its source sequence and whole-case context. Compare the library evaluation with the matching observations and note what a missing endpoint means. The default witness is an investigation example, not a representative sample or proof of causation. To examine another case, pass its identifier to `case_evidence` without filtering away other events.

In [ ]:
case_witnesses = {}
for name, run in scored_runs.items():
    witness = case_evidence(run)
    case_witnesses[name] = witness
    display(Markdown(f"### Full source case: {name}"))
    for label, value in witness.items():
        display(Markdown(f"**{label}**"))
        display(value)

### Place the case evidence on a timeline

This figure uses event timestamps returned by the library for a complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane keeps the remaining recorded activities visible. Coincident points can overlap and establish no order. Undated events remain in the source table and cannot be placed on the time axis. The selected case is an investigation example, not a representative sample; return to its full source witness before interpreting an apparent gap.

In [ ]:
from bpic_analysis import plot_case_timeline
for identifier, run in runs.items():
    if "result" not in run:
        continue
    figure = plot_case_timeline(run)
    display(figure)
    plt.close(figure)

## 12. Validate the result and decide what it supports

A traveller-facing completeness check is appropriate only where resubmission concerns avoidable missing information. A reviewer backup route is appropriate only where a valid claim waits for an unavailable authorised reviewer. Payment queue changes require evidence that approval was complete. Review these mechanisms separately and preserve alternative legitimate approval routes.

The integrity check records norm round-trip consistency, score decomposition, assessed case-index agreement and the retained event count. It does not independently certify extraction completeness, historical field validity or coverage of the real process. A failed check or incomplete binding limits the conclusion even when a plot looks convincing. Keep unsupported checks and cases without usable evidence in the journal.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(integrity(run))
display(journal(runs))

## 13. Process improvement and support: choose a feasible pilot

The proposals below are conditional on the case review. They change the business process, not the software package. Select one supported mechanism, a responsible owner and comparable cases. Agree the outcome and quality safeguard before starting. An improved WISE score alone is not a business benefit.

In [ ]:
process_improvements = pd.DataFrame([{'Process issue to confirm': 'Claims are returned for recurring missing or inconsistent information', 'Pilot change': 'Add a document-specific completeness check and clear correction reason before submission', 'Owner': 'Travel administration lead', 'Success measure': 'Fewer avoidable resubmissions and faster complete decisions within comparable document types', 'Safeguard': 'Preserve necessary rejection and legitimate withdrawal; do not favour unsupported approvals'}, {'Process issue to confirm': 'Complete requests wait for an unavailable authorised reviewer', 'Pilot change': 'Pilot an approved backup reviewer route with a visible accepting owner', 'Owner': 'Budget-owner coordinator', 'Success measure': 'Shorter complete-request approval intervals with stable authorised-review coverage', 'Safeguard': 'Retain required role checks, legitimate director routes and applicable separation of duties'}, {'Process issue to confirm': 'Fully approved requests wait unnecessarily for payment handling', 'Pilot change': 'Review payment queue ownership and pilot a predictable handoff to finance', 'Owner': 'Accounts payable lead', 'Success measure': 'Shorter approval-to-handling intervals and verified settlement where that evidence exists', 'Safeguard': 'Do not infer settlement or duplicate-payment correctness from handling events alone'}])
display(process_improvements)

## 14. Full analysis journal and next iteration

The execution journal records what was actually loaded and assessed. The decision journal records the remaining business review and pilot decisions. Replace the pending notes after a stakeholder review; they do not imply that an organisation has approved a norm or that a pilot has occurred. If a case changes the explanation, return to the relevant PM² stage and rerun the same complete-case assessment.

Saved tables and figures may contain case identifiers or business context. Export is optional and disabled initially; keep shared results consistent with the source’s permitted use.

In [ ]:
decision_journal = pd.DataFrame([
    {"PM² stage": "Planning", "Decision": "Confirm the business goal, acceptable outcomes and policy owner", "Evidence to record": "Owner-approved scope and diagnostic cut points", "Status": "Pending business review"},
    {"PM² stage": "Extraction", "Decision": "Confirm source identity, observation boundary and case unit", "Evidence to record": "Execution journal and source identifiers", "Status": "Review computed journal"},
    {"PM² stage": "Data processing", "Decision": "Retain whole cases and review changing context or unavailable endpoints", "Evidence to record": "Context profiles and full-case witnesses", "Status": "Review computed evidence"},
    {"PM² stage": "Mining and analysis", "Decision": "Choose a lead that survives meaningful preference and threshold scenarios", "Evidence to record": "Coverage, signed drivers and shortlist sensitivity", "Status": "Review computed comparisons"},
    {"PM² stage": "Evaluation", "Decision": "Distinguish avoidable work, necessary control and recording gaps", "Evidence to record": "Source-case explanation and integrity checks", "Status": "Pending case interpretation"},
    {"PM² stage": "Process improvement and support", "Decision": "Approve one feasible pilot with a business outcome and stopping rule", "Evidence to record": "Owner, comparable cases, quality safeguards and follow-up date", "Status": "No pilot undertaken"},
])
display(journal(runs))
display(decision_journal)
if EXPORT_RESULTS:
    save_outputs(runs, Path("results") / "BPIC2020")